# Project 4 — Trợ lý hỏi–đáp RAG trên văn bản pháp quy FMCG
## Phần 2 — Vector hóa (Embedding) & Truy hồi ngữ nghĩa

**Bối cảnh.** Trợ lý trả lời câu hỏi về quy định pháp luật áp dụng cho vận hành chuỗi cung ứng FMCG: nhập nguyên liệu (hải quan) → sản xuất (an toàn thực phẩm) → ra thị trường (ghi nhãn). Yêu cầu cốt lõi: câu trả lời **kèm trích dẫn nguồn** đến từng Điều/Khoản của văn bản gốc.

**Đầu vào.** `data/chunks.json` — 291 chunk đã cắt theo cấu trúc Điều/Khoản ở Phần 1, mỗi chunk có trường `text_for_embedding` (nội dung kèm tiền tố tiêu đề Điều cha) và metadata phục vụ trích dẫn.

**Nội dung notebook này.**
1. **Vector hóa** toàn bộ chunk bằng mô hình embedding đa ngữ `intfloat/multilingual-e5-base` (chuẩn hóa L2 để cosine similarity rút gọn thành tích vô hướng).
2. **Đánh giá định tính chất lượng truy hồi:** bộ 5 câu hỏi thử (4 trong phạm vi tài liệu + 1 ngoài phạm vi), truy hồi top-5 và kiểm tra thủ công độ liên quan cùng phân bố điểm tương đồng.
3. **Lưu ma trận embedding** ra đĩa để tái sử dụng ở Phần 3, tránh mã hóa lại.

**Đầu ra:** `data/embeddings.npy`

> Mô hình embedding chạy local trên GPU (NVIDIA RTX 4050). Chi tiết lý thuyết: xem `Tong_hop_kien_thuc_4.md` (mục II, IV).

### 1. Vector hóa toàn bộ chunk

Áp tiền tố `"passage: "` cho tài liệu theo yêu cầu của dòng e5 (tác vụ truy hồi là *asymmetric*: câu hỏi ngắn ↔ đoạn văn dài). Chuẩn hóa vector về độ dài 1 (`normalize_embeddings=True`) để cosine similarity = tích vô hướng.

In [1]:
import json
import torch
import numpy as np
from sentence_transformers import SentenceTransformer


file_path = 'data/chunks.json'
with open(file_path, 'r', encoding='utf-8') as f:
    chunks = json.load(f)

print(f"Đã tải thành công {len(chunks)} chunks.")

Đã tải thành công 291 chunks.


In [2]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Đang khởi chạy mô hình trên: {device.upper()}")

Đang khởi chạy mô hình trên: CUDA


In [3]:
model = SentenceTransformer('intfloat/multilingual-e5-base', device=device)

texts = ["passage: " + c["text_for_embedding"] for c in chunks]
emb = model.encode(texts, batch_size=32, normalize_embeddings=True, show_progress_bar=True)

emb.shape

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/10 [00:00<?, ?it/s]

(291, 768)

In [4]:
np.linalg.norm(emb[0])

np.float32(1.0)

### 2. Đánh giá định tính chất lượng truy hồi

Truy hồi bằng brute-force (tích vô hướng với toàn bộ 291 vector — chính xác tuyệt đối, đủ nhanh ở quy mô này). Bộ câu hỏi thử được thiết kế có chủ đích: câu định nghĩa, câu thủ tục, câu ghi nhãn, câu hải quan, và **một câu ngoài phạm vi tài liệu** để quan sát phân bố điểm khi không có dữ liệu phù hợp.

Áp tiền tố `"query: "` cho câu hỏi.

In [5]:
def search(query, top_k=5):
    q = model.encode(["query: " + query], normalize_embeddings=True)  # shape (1, 768)
    scores = emb @ q[0]          # tích vô hướng = cosine vì đã chuẩn hóa
    idx = np.argsort(-scores)[:top_k]
    return [(chunks[i], float(scores[i])) for i in idx]

queries = ["thực phẩm bao gói sẵn là gì",
           "hồ sơ tự công bố sản phẩm gồm những gì",
           "nhãn hàng hóa nhập khẩu bắt buộc ghi những nội dung nào",
           "thời hạn nộp tờ khai hải quan",
           "nhân viên nghỉ thai sản mấy ngày"]

for query in queries:
    results = search(query, top_k=5)
    print('\n', '='*50 , sep='')
    print(f'Kết quả retrieval của câu hỏi "{query}" là:')
    print('='*50 , sep='')
    for i, result in enumerate(results):
        print(f'({i+1}) Chunk id:', result[0]['chunk_id'])
        print('Nội dung:', result[0]['text_for_embedding'])
        print('Độ tương đồng:', result[1], end='\n \n')


Kết quả retrieval của câu hỏi "thực phẩm bao gói sẵn là gì" là:
(1) Chunk id: 61/VBHN-VPQH_Điều_2_part_3
Nội dung: Điều 2. Giải thích từ ngữ:
22. Thực phẩm tăng cường vi chất dinh dưỡng là thực phẩm được bổ sung vitamin, chất khoáng, chất vi lượng nhằm phòng ngừa, khắc phục sự thiếu hụt các chất đó đối với sức khỏe cộng đồng hay nhóm đối tượng cụ thể trong cộng đồng.

23. Thực phẩm chức năng là thực phẩm dùng để hỗ trợ chức năng của cơ thể con người, tạo cho cơ thể tình trạng thoải mái, tăng sức đề kháng, giảm bớt nguy cơ mắc bệnh, bao gồm thực phẩm bổ sung, thực phẩm bảo vệ sức khỏe, thực phẩm dinh dưỡng y học.

24. Thực phẩm biến đổi gen là thực phẩm có một hoặc nhiều thành phần nguyên liệu có gen bị biến đổi bằng công nghệ gen.

25. Thực phẩm đã qua chiếu xạ là thực phẩm đã được chiếu xạ bằng nguồn phóng xạ để xử lý, ngăn ngừa sự biến chất của thực phẩm.

26. Thức ăn đường phố là thực phẩm được chế biến dùng để ăn, uống ngay, trong thực tế được thực hiện thông qua hình thức bán ron

### 3. Lưu ma trận embedding

Lưu `embeddings.npy` để Phần 3 (sinh câu trả lời) nạp trực tiếp, không mã hóa lại.

In [6]:
np.save('data/embeddings.npy', emb)
print(f"Đã lưu thành công các vector embedding vào file: data/embeddings.npy")

Đã lưu thành công các vector embedding vào file: data/embeddings.npy
